Samuel Scott - Module 3 Assignment 3.1: Use Case - Taxi Cancellations

# Part 1: Business Situation
The Company
In late 2013, the taxi company Yourcabs.com in Bangalore, India was facing a problem with the drivers
using its platform. Not all drivers were showing up for their scheduled calls. Drivers would cancel their
acceptance of a call, and if the cancellation did not occur with adequate notice, the customer would be
delayed or left without a ride entirely.
Bangalore is a key tech center in India, and technology was actively transforming the taxi industry at the
time. Yourcabs.com featured an online booking system (though customers could also phone in) and
presented itself as a taxi booking portal. The Uber ride-sharing service began its Bangalore operations in
mid-2014, intensifying competitive pressure.

## The Data
Yourcabs.com collected booking data from 2011 to 2013 and posted a Kaggle contest, in coordination
with the Indian School of Business, to learn what it could about the cancellation problem.
The data for this case is a randomly selected subset of the original data, with 10,000 rows (one row per
booking) and 17 input variables. These include user (customer) ID, vehicle model, whether the booking
was made online or via the mobile app, type of travel, type of booking package, geographic information,
and the date and time of the scheduled trip.
The target variable is a binary indicator of whether a ride was canceled. The overall cancellation rate is
between 7% and 8%, which means this is an imbalanced classification problem. Plan your evaluation
strategy accordingly: accuracy alone will not tell you whether a model is operationally useful.

In [1]:
#!pip install keras
#!pip install tensorflow

In [21]:
#imports
import mlba
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import tensorflow as tf
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

## Sanity Check

In [3]:
df = pd.read_csv('C:/Users/samsc/Desktop/ADS-505/Taxi-cancellation-case.csv')
df.head()

,row#,user_id,vehicle_model_id,package_id,travel_type_id,from_area_id,to_area_id,from_city_id,to_city_id,from_date,to_date,online_booking,mobile_site_booking,booking_created,from_lat,from_long,to_lat,to_long,Car_Cancellation
0,1,17712,12,NaN,2,1021.0,1323.0,NaN,NaN,1/1/13 22:33,NaN,0,0,1/1/13 8:01,13.028530,77.54625,12.869805,77.653211,0
1,2,17037,12,NaN,2,455.0,1330.0,NaN,NaN,1/1/13 12:43,NaN,0,0,1/1/13 9:59,12.999874,77.67812,12.953434,77.706510,0
2,3,761,12,NaN,2,814.0,393.0,NaN,NaN,1/2/13 0:28,1/3/13 0:00,1,0,1/1/13 12:14,12.908993,77.68890,13.199560,77.706880,0
3,4,868,12,NaN,2,297.0,212.0,NaN,NaN,1/1/13 13:12,NaN,0,0,1/1/13 12:42,12.997890,77.61488,12.994740,77.607970,0
4,5,21716,28,NaN,2,1237.0,330.0,NaN,NaN,1/1/13 16:33,NaN,0,0,1/1/13 15:07,12.926450,77.61206,12.858833,77.589127,0


In [4]:
df.shape

(10000, 19)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   row#                 10000 non-null  int64  
 1   user_id              10000 non-null  int64  
 2   vehicle_model_id     10000 non-null  int64  
 3   package_id           1752 non-null   float64
 4   travel_type_id       10000 non-null  int64  
 5   from_area_id         9985 non-null   float64
 6   to_area_id           7909 non-null   float64
 7   from_city_id         3706 non-null   float64
 8   to_city_id           339 non-null    float64
 9   from_date            10000 non-null  object 
 10  to_date              5822 non-null   object 
 11  online_booking       10000 non-null  int64  
 12  mobile_site_booking  10000 non-null  int64  
 13  booking_created      10000 non-null  object 
 14  from_lat             9985 non-null   float64
 15  from_long            9985 non-null   

### Question 1 - Predictive Use Case
How can a predictive model based on these data be used by Yourcabs.com? Identify the chief
stakeholder, the action the model output would trigger, and the asymmetric cost of false positives versus
false negatives in this business context.

A predictive model would be very inciteful for the Yourcabs.com business as it would help make discoveries on what attributes contribute most to cancel taxi rides. The chief stakeholder would be the company website Yourcabs.com as it is responsible for recording records of all taxi rides. A predictive model would trigger outputs that provide both results of car_cancellations so it is able to learn what rides are more likely to be cancelled, helping businesses find customers to pick up that will more likely not cancel, therefore improving efficiency and profit.

Assymetric cost of false positives vs false negative:
In the context of the taxi business (0 : No Cancel, 1: Cancel):
-True Positive means that the customer actual cancel and the model predicts cancellation (important that our model can correctly identify cancellations)
-True Negative means that the customer does not cancel and the model predicts they don't cancel (the least important of the 4 boxes because in a business context finding cancellations is what is lowering profit gains, but still important that the model understands and correctly identifies this)
-False Negative is that they cancel but the model predicts no cancel (most important box for correctness, as this is the most harmful for revenue gain)
-False Positive is that they don't cancel but the model predicts cancel (could be bad because we are denying customer pickups that could make the business revenue)

### Question 2 - Profiling Use Case
How can a profiling model (one that identifies the predictors that distinguish canceled from uncanceled
trips) be used by Yourcabs.com? Note that predictive and profiling use cases often call for different
models and different evaluation criteria.

The profiling model focuses more on current data results while the predictive model focuses more on how a new record will be classified. The profiling model will answer questions more about how current data what predictors are consistently resulting to ride share cancellations? Attributes based on the Yourcabs.com database that could be strong contributors could be location of pick up and arrival, the data, online_booking versus other types of booking. 


### Question 3 - Explore, Prepare, and Transform the Data
Explore, prepare, and transform the data to facilitate predictive modeling. The hints below identify the key
decisions you will need to make and document.

#### 3.1 Move to an Initial Model Quickly
In exploratory modeling, it is useful to reach an initial model fairly soon, before resolving every data
preparation issue. For example, the GPS information is one such issue you could defer (other geographic
information is available, so you can come back to GPS later).

In [6]:
#What features are needed for the initial model? I assume based on the question to remove GPS lat and long coordinates
#drop columns that have NAs too

#Code that helped me decipher what columns to remove
# initial_predictors = df.drop(columns=['from_lat', 'from_long', 'to_lat', 'to_long', 'package_id', 'from_area_id',
#                                       'to_area_id', 'from_city_id', 'to_city_id', 'to_date']) #all of these are missing data columns
# initial_predictors.info()
# initial_predictors = initial_predictors.drop(columns=['row#', 'user_id', 'vehicle_model_id', 'travel_type_id', 'Car_Cancellation']) #id columns that
#don't contribute to predictive results

#These two columns since they are date formatted need to be encoded or adjusted to fit the model. Since it is baseline model these
#predictors were not needed'booking_created', 'from_date'

outcome = 'Car_Cancellation'
predictors = ['online_booking', 'mobile_site_booking']

In [7]:
#Since LDA works well for smaller samples I will use it as a baseline model. 
X = df[predictors]
y = df[outcome]
da = LinearDiscriminantAnalysis()
da.fit(X, y)
mlba.classificationSummary(y_true=y, y_pred=da.predict(X))

Confusion Matrix (Accuracy 0.9257)

       Prediction
Actual    0    1
     0 9257    0
     1  743    0


Baseline accuracy is approximately 93%. The baseline model made no predictions on canceled reservations because most of the car_cancellations are not usually canceled which makes sense because most customers intend to use the taxi if they request for a taxi. 

#### 3.2 Handle Missing Data
How will you deal with missing data, including cases where NaN is indicated? Document your imputation,
deletion, or encoding strategy and your rationale for each variable.

-In fact, in general, id types are not useful for prediction. They are only useful for labeling unique values/records. However vehicle_model_id has only several unique values, since none of the values are missing this could also proved to lead to some predictive power.
-The to_date column has many missing values so it was dropped as well. Most of the columns were deleted for prediction and profiling purposes.

In [8]:
df = df.drop(columns=['row#','user_id','package_id','travel_type_id', 'from_area_id', 'to_area_id', 'from_city_id', 'to_city_id', 'to_date'])

In [9]:
df.info() #checking new dataframe after dropping id columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   vehicle_model_id     10000 non-null  int64  
 1   from_date            10000 non-null  object 
 2   online_booking       10000 non-null  int64  
 3   mobile_site_booking  10000 non-null  int64  
 4   booking_created      10000 non-null  object 
 5   from_lat             9985 non-null   float64
 6   from_long            9985 non-null   float64
 7   to_lat               7909 non-null   float64
 8   to_long              7909 non-null   float64
 9   Car_Cancellation     10000 non-null  int64  
dtypes: float64(4), int64(4), object(2)
memory usage: 781.4+ KB


Lastly we have missing values in both latitude and longitude distances. In the context of this data set since the cab pickups are from Bangalore,
India, a place where the longitude and latitude values will not change drastically as oppose to having the company track taxi pickups from New York City, USA, London, U.K. and Bangalore, India, it is reasonable to impute the missing longitude and latitude values by median. It is also reasoanably since a majority of the values are not missing from these four columns. Therefore imputing the data is a safe approach in this instance. 

In [10]:
df['from_lat'] = df['from_lat'].fillna(df['from_lat'].median())
df['from_long'] = df['from_long'].fillna(df['from_long'].median())
df['to_lat'] = df['to_lat'].fillna(df['to_lat'].median())
df['to_long'] = df['to_long'].fillna(df['to_long'].median())
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   vehicle_model_id     10000 non-null  int64  
 1   from_date            10000 non-null  object 
 2   online_booking       10000 non-null  int64  
 3   mobile_site_booking  10000 non-null  int64  
 4   booking_created      10000 non-null  object 
 5   from_lat             10000 non-null  float64
 6   from_long            10000 non-null  float64
 7   to_lat               10000 non-null  float64
 8   to_long              10000 non-null  float64
 9   Car_Cancellation     10000 non-null  int64  
dtypes: float64(4), int64(4), object(2)
memory usage: 781.4+ KB


### 3.3 Engineer Features from Date and Time
Think about what useful information is held within the date and time fields (the booking timestamp and the
trip timestamp). Consider features such as lead time between booking and trip, hour of day, day of week,
weekend versus weekday, and whether the booking timing itself signals cancellation risk.

In [11]:
#from_date - time stamp of requested trip start
#booking_created - time stamp of booking
#the booking always happens before the from_date

#Features to create:

#lead_time (in hours) feature which is from_date - booking_time difference
#extract hour of day from booking and from_date (done)
#day_of_week I assume is similar process (done)
#weekend vs weekday (create feature and classify it as either weekday or weekend
#booking_time feature

#type(df['from_date'][0]) #checking the value data types for feature
#type(df['booking_created'][0]) #checking the value data types for feature


In [12]:
#Inspiration behind the engineered features:
#https://pandas.pydata.org/docs/reference/api/pandas.Timestamp.html

#Convert booking month to Pandas timestamp:
df['booking_created'] = pd.to_datetime(df['booking_created'])
#type(df['booking_created'][0]) 

#creating day_of_week booking feature
df['booking_day_of_week'] = df['booking_created'].dt.day_of_week
#df['booking_day_of_week'] (days of week range from 0-6)

#Same process but for the from_date feature
#Convert booking month to Pandas timestamp:
df['from_date'] = pd.to_datetime(df['from_date'])

#creating day_of_week booking feature
df['from_date_day_of_week'] = df['from_date'].dt.day_of_week

C:\Users\samsc\AppData\Local\Temp\ipykernel_10532\3426982220.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['booking_created'] = pd.to_datetime(df['booking_created'])
C:\Users\samsc\AppData\Local\Temp\ipykernel_10532\3426982220.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['from_date'] = pd.to_datetime(df['from_date'])


<class 'pandas._libs.tslibs.timestamps.Timestamp'>


0       1
1       1
2       2
3       1
4       1
       ..
9995    6
9996    0
9997    4
9998    2
9999    5
Name: from_date_day_of_week, Length: 10000, dtype: int32

In [13]:
#booking_created hour feature
df['booking_created_hour'] = df['booking_created'].dt.hour
#df['booking_created_hour']

#from_date hour feature
df['from_date_hour'] = df['from_date'].dt.hour
#df['from_date_hour']

In [14]:
#month features
df['booking_created_month'] = df['booking_created'].dt.month
df['from_date_month'] = df['from_date'].dt.month

In [15]:
#lead time variable function (convert lead time to hours)
df['lead_time'] = ((df['from_date'] - df['booking_created']).dt.total_seconds() / 3600).round(1).astype(int) 
#60 seconds in a minute times 60 seconds in an hour

In [16]:
#determine if the from_date is a weekend date?
df['from_date_weekend'] = df['from_date_day_of_week'].isin([5, 6]) #gives True and False results

In [17]:
#determine if the booking_date is a weekend date?
df['booking_weekend'] = df['booking_day_of_week'].isin([5, 6])

In [18]:
#drop columns that help create our engineered features
df = df.drop(columns=['from_date', 'booking_created'])
df.head()

,vehicle_model_id,online_booking,mobile_site_booking,from_lat,from_long,to_lat,to_long,Car_Cancellation,booking_day_of_week,from_date_day_of_week,booking_created_hour,from_date_hour,booking_created_month,from_date_month,lead_time,from_date_weekend,booking_weekend
0,12,0,0,13.028530,77.54625,12.869805,77.653211,0,1,1,8,22,1,1,14,False,False
1,12,0,0,12.999874,77.67812,12.953434,77.706510,0,1,1,9,12,1,1,2,False,False
2,12,1,0,12.908993,77.68890,13.199560,77.706880,0,1,2,12,0,1,1,12,False,False
3,12,0,0,12.997890,77.61488,12.994740,77.607970,0,1,1,12,13,1,1,0,False,False
4,28,0,0,12.926450,77.61206,12.858833,77.589127,0,1,1,15,16,1,1,1,False,False


#### 3.4 Handle Categorical Variables
Think about the categorical variables and how to encode them. Should you turn them all into dummies?
Use only some? High-cardinality categoricals (like user ID and vehicle model code) require particular
care.

Using vehicle_model_id and encoded that sounds useful because maybe a particular car brand is more likely to lead to cancelled taxi rides. And the id values do not mean anything therfore this variable will be encoded. 

In [23]:
vehicle_model_id = df[['vehicle_model_id']]
cat_encoder = OneHotEncoder()
vehicle_model_id_one_hot = cat_encoder.fit_transform(vehicle_model_id)

In [27]:
#make a dataframe with a sparse matrix, column names and index
vehicle_model_id_one_hot_df = pd.DataFrame(
    vehicle_model_id_one_hot.toarray(),
    columns=cat_encoder.get_feature_names_out(['vehicle_model_id']),
    index=df.index
)

In [28]:
df = pd.concat([df, vehicle_model_id_one_hot_df], axis=1)

In [31]:
df = df.drop(columns=['vehicle_model_id']) # no need for this column anymore!
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 36 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   online_booking         10000 non-null  int64  
 1   mobile_site_booking    10000 non-null  int64  
 2   from_lat               10000 non-null  float64
 3   from_long              10000 non-null  float64
 4   to_lat                 10000 non-null  float64
 5   to_long                10000 non-null  float64
 6   Car_Cancellation       10000 non-null  int64  
 7   booking_day_of_week    10000 non-null  int32  
 8   from_date_day_of_week  10000 non-null  int32  
 9   booking_created_hour   10000 non-null  int32  
 10  from_date_hour         10000 non-null  int32  
 11  booking_created_month  10000 non-null  int32  
 12  from_date_month        10000 non-null  int32  
 13  lead_time              10000 non-null  int64  
 14  from_date_weekend      10000 non-null  bool   
 15  boo

#### Question 4 - Fit Predictive Models
Fit several predictive models of your choice. Do these models provide information on how the predictor
variables relate to cancellations? Compare what each model reveals (or hides) about variable importance
and interactions.

In [ ]:
#Logistic Regression

In [ ]:
#KNearestNeighbor

In [ ]:
#Neural Net

#### Question 5 - Predictive Performance: Error Rates
Report the predictive performance of your model in terms of error rates using the confusion matrix. How
well does the model perform? Given the 7 to 8 percent cancellation rate, what does accuracy alone hide?

#### Question 6 - Predictive Performance: Ranking (Lift)
Examine the predictive performance of your model in terms of ranking using a lift chart or cumulative
gains chart. How much better than random does the model do at concentrating actual cancellations in the
top-ranked bookings? At what cutoff would it be useful operationally (for example, flagging the top 10% or
20% of bookings for intervention)? Can the model be used in practice?